In [1]:
# ✅ INSTALL DEPENDENCIES (Colab-compatible)
!pip install -q datasets transformers accelerate bitsandbytes faiss-cpu xformers peft torchvision sentence-transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 70.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.4/43.4 MB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 55.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 85.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
# ✅ INSTALL DEPENDENCIES (Colab-compatible)

# ✅ IMPORTS
from datasets import load_dataset
from PIL import Image
import torch
import numpy as np
import faiss
import random
from collections import Counter
import os
import torch.nn.functional as F
from transformers import (
    BitsAndBytesConfig, AutoProcessor, LlavaForConditionalGeneration,
    CLIPProcessor, CLIPModel
)
from sentence_transformers import SentenceTransformer

random.seed(42)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

# ✅ JS Divergence
def js_divergence(p, q):
    p = torch.tensor(p, dtype=torch.float32)
    q = torch.tensor(q, dtype=torch.float32)
    p = F.softmax(p, dim=0)
    q = F.softmax(q, dim=0)
    m = 0.5 * (p + q)
    kl_pm = F.kl_div(m.log(), p, reduction='sum')
    kl_qm = F.kl_div(m.log(), q, reduction='sum')
    return 0.5 * (kl_pm + kl_qm).item()

# ✅ CLIP Text-Image Retriever
class CLIPTextImageRetriever:
    def __init__(self):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.clip_model_id = "openai/clip-vit-large-patch14"
        self.text_model_id = "sentence-transformers/clip-ViT-L-14"
        self.clip_processor = CLIPProcessor.from_pretrained(self.clip_model_id)
        self.clip_model = CLIPModel.from_pretrained(self.clip_model_id).to(self.device)
        self.st_model = SentenceTransformer(self.text_model_id).to(self.device)
        self.text_data, self.image_data, self.embeddings, self.metadata = [], [], [], []
        self.index = None

    def encode_texts(self, texts):
        return self.st_model.encode(texts, normalize_embeddings=True)

    def encode_images(self, images):
        inputs = self.clip_processor(images=images, return_tensors="pt", padding=True).to(self.device)
        with torch.no_grad():
            feats = self.clip_model.get_image_features(**inputs)
            feats = feats / feats.norm(dim=-1, keepdim=True)
        return feats.cpu().numpy()

    def add_texts(self, texts):
        for text in texts:
            self.text_data.append(text)
            emb = self.encode_texts([text])[0]
            self.embeddings.append(emb)
            self.metadata.append(("text", len(self.text_data) - 1))

    def add_images(self, images):
        for img in images:
            if img is None:
                continue
            self.image_data.append(img)
            img_idx = len(self.image_data) - 1
            img_emb = self.encode_images([img])[0]
            self.embeddings.append(img_emb)
            self.metadata.append(("image", img_idx))

    def build_index(self):
        if self.embeddings:
            all_embs = np.vstack(self.embeddings).astype("float32")
            self.index = faiss.IndexFlatIP(all_embs.shape[1])
            self.index.add(all_embs)

    def encode_query(self, query_text=None, query_image=None):
        if query_text and query_image:
            text_emb = self.encode_texts([query_text])
            image_emb = self.encode_images([query_image])
            combined = text_emb + image_emb
            return combined / np.linalg.norm(combined, axis=1, keepdims=True)
        elif query_text:
            return self.encode_texts([query_text])
        elif query_image:
            return self.encode_images([query_image])
        else:
            raise ValueError("Provide text and/or image as query.")

    def retrieve(self, query_text=None, query_image=None, top_k=10):
        q_emb = self.encode_query(query_text, query_image)
        scores, indices = self.index.search(q_emb, top_k)
        ctx_texts, ctx_images = [], []
        for i in indices[0]:
            doc_type, doc_idx = self.metadata[i]
            if doc_type == "text":
                ctx_texts.append(self.text_data[doc_idx])
            else:
                ctx_texts.append("[Image]")
                ctx_images.append(self.image_data[doc_idx])
        return ctx_texts, ctx_images

# ✅ Generator
class MultimodalGenerator:
    def __init__(self, model_name="llava-hf/llava-1.5-7b-hf"):
        self.processor = AutoProcessor.from_pretrained(model_name)
        self.model = LlavaForConditionalGeneration.from_pretrained(
            model_name, quantization_config=quantization_config, device_map="auto")

    def generate(self, query, query_image=None, context_texts=[], context_images=[]):
        img_list, question_string = [], query
        if query_image:
            question_string = f"<image>\n{query}"
            img_list.append(query_image)
        ctx = "\n".join(context_texts)
        if context_images:
            ctx = f"{'<image>'*len(context_images)}\n{ctx}"
            img_list.extend(context_images)
        prompt = f"USER: Context:{ctx}\n\nQuestion: {question_string}\n\nASSISTANT:"
        inputs = self.processor(img_list if img_list else prompt, prompt, padding=True, return_tensors="pt").to("cuda")
        output = self.model.generate(**inputs, max_new_tokens=200)
        return self.processor.batch_decode(output, skip_special_tokens=True)[0].split("ASSISTANT:")[-1]

# ✅ CLIP-based RAG Wrapper
class CLIPMultimodalRAG:
    def __init__(self):
        self.retriever = CLIPTextImageRetriever()
        self.generator = MultimodalGenerator()

    def add_texts(self, texts): self.retriever.add_texts(texts)
    def add_images(self, images): self.retriever.add_images(images)
    def build_index(self): self.retriever.build_index()

    def query(self, query_text=None, query_image=None, top_k=10):
        ctx_texts, ctx_images = self.retriever.retrieve(query_text, query_image, top_k)
        return self.generator.generate(query=query_text, query_image=query_image, context_texts=ctx_texts, context_images=ctx_images)


In [3]:
k_ar=[0, 3,5,10,15,20]
acc_simp_l=[]
top_3_acc_l=[]

In [4]:
# ✅ Load dataset
val_data = load_dataset("merve/vqav2-small", split="validation[:150]")

# ✅ Build knowledge base
rag_texts = [f"Q: {ex['question']}\nA: {ex['multiple_choice_answer']}" for ex in val_data]
rag_images = [ex['image'].convert("RGB") for ex in val_data]

rag = CLIPMultimodalRAG()
rag.add_texts(rag_texts)
rag.add_images(rag_images)
rag.build_index()

# ✅ Select 100 for evaluation
indices = random.sample(range(len(val_data)), 100)
test_questions = [val_data[i]['question'] for i in indices]
test_answers = [val_data[i]['multiple_choice_answer'] for i in indices]
test_images = [val_data[i]['image'].convert("RGB") for i in indices]


# ✅ Accuracy

def compute_simple_accuracy(preds, refs):
    correct = sum(p == r.lower() for p, r in zip(preds, refs))
    return correct / len(refs) if refs else 0.0


def compute_topk_accuracy(preds, refs, k=3):
    total, correct = 0, 0
    for pred, ref in zip(preds, refs):
        ref = ref.strip().lower()
        guesses = [g.strip().lower() for g in pred.split(",")]
        guesses = list(dict.fromkeys(guesses))
        top_k = guesses[:k]
        if ref in top_k:
            correct += 1
        total += 1
    return correct / total if total > 0 else 0.0


# ✅ Evaluate without RAG
gen = MultimodalGenerator()
noctx_preds = [
    gen.generate(query=q, query_image=img).strip().lower()
    for q, img in zip(test_questions, test_images)
]

acc_simp_l.append(compute_simple_accuracy(noctx_preds, test_answers)*100)
top_3_acc_l.append(compute_topk_accuracy(noctx_preds, test_answers, k=3)*100)
# ✅ Report
print(f"✅ Non-RAG Accuracy: {compute_simple_accuracy(noctx_preds, test_answers)*100:.2f}%")
print(f"✅ Non-RAG Top-3 Accuracy: {compute_topk_accuracy(noctx_preds, test_answers, k=3)*100:.2f}%")

for k in k_ar[1:]:
  # ✅ Evaluate with RAG
  rag_preds = [
      rag.query(query_text=q, query_image=img,top_k=k ).strip().lower()
      for q, img in zip(test_questions, test_images)
  ]





  acc_simp_l.append(compute_simple_accuracy(rag_preds, test_answers)*100)
  top_3_acc_l.append(compute_topk_accuracy(rag_preds, test_answers, k=3)*100)
  # ✅ Report
  print(f"✅ RAG Accuracy: {compute_simple_accuracy(rag_preds, test_answers)*100:.2f}%")
  print(f"✅ RAG Top-3 Accuracy: {compute_topk_accuracy(rag_preds, test_answers, k=3)*100:.2f}%")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/403 [00:00<?, ?B/s]

validation-00000-of-00007.parquet:   0%|          | 0.00/484M [00:00<?, ?B/s]

validation-00001-of-00007.parquet:   0%|          | 0.00/486M [00:00<?, ?B/s]

validation-00002-of-00007.parquet:   0%|          | 0.00/483M [00:00<?, ?B/s]

...

model-00002-of-00003.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

✅ Non-RAG Accuracy: 0.00%
✅ Non-RAG Top-3 Accuracy: 31.00%
✅ RAG Accuracy: 21.00%
✅ RAG Top-3 Accuracy: 53.00%
✅ RAG Accuracy: 19.00%
✅ RAG Top-3 Accuracy: 52.00%
✅ RAG Accuracy: 24.00%
✅ RAG Top-3 Accuracy: 53.00%
✅ RAG Accuracy: 20.00%
✅ RAG Top-3 Accuracy: 48.00%
✅ RAG Accuracy: 22.00%
✅ RAG Top-3 Accuracy: 45.00%


In [5]:
acc_simp_l

[0.0, 21.0, 19.0, 24.0, 20.0, 22.0]

In [6]:
top_3_acc_l

[31.0, 53.0, 52.0, 53.0, 48.0, 45.0]

In [7]:
k_ar=[0,3,5,10,15,20]
acc_simp_l_cap=[]
top_3_acc_l_cap=[]

In [8]:
from transformers import  BlipProcessor, BlipForConditionalGeneration


class LightweightBLIPCaptioner:
    def __init__(self, model_name="Salesforce/blip-image-captioning-base", device=None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.processor = BlipProcessor.from_pretrained(model_name)
        self.model = BlipForConditionalGeneration.from_pretrained(model_name).to(self.device)

    def caption(self, image):
        inputs = self.processor(images=image, return_tensors="pt").to(self.device)
        with torch.no_grad():
            out = self.model.generate(**inputs, max_new_tokens=64)
        return self.processor.batch_decode(out, skip_special_tokens=True)[0]

# ✅ CLIP Text-Image Retriever
class CLIPTextImageRetriever:
    def __init__(self):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.clip_model_id = "openai/clip-vit-large-patch14"
        self.text_model_id = "sentence-transformers/clip-ViT-L-14"
        self.clip_processor = CLIPProcessor.from_pretrained(self.clip_model_id)
        self.clip_model = CLIPModel.from_pretrained(self.clip_model_id).to(self.device)
        self.st_model = SentenceTransformer(self.text_model_id).to(self.device)
        self.text_data, self.image_data, self.image_captions, self.embeddings, self.metadata = [], [], [], [], []
        self.index = None
        self.captioner = LightweightBLIPCaptioner()

    def encode_texts(self, texts):
        return self.st_model.encode(texts, normalize_embeddings=True)

    def encode_images(self, images):
        inputs = self.clip_processor(images=images, return_tensors="pt", padding=True).to(self.device)
        with torch.no_grad():
            feats = self.clip_model.get_image_features(**inputs)
            feats = feats / feats.norm(dim=-1, keepdim=True)
        return feats.cpu().numpy()

    def add_texts(self, texts):
        for text in texts:
            self.text_data.append(text)
            emb = self.encode_texts([text])[0]
            self.embeddings.append(emb)
            self.metadata.append(("text", len(self.text_data) - 1))

    def add_images(self, images, captions=None):
        if self.captioner:
            captions = [caption if caption else self.captioner.caption(img) for img, caption in zip(images, captions or [None] * len(images))]
        else:
            captions = captions or [None] * len(images)

        for img, caption in zip(images, captions):
            if img is None:
                continue
            self.image_data.append(img)
            img_idx = len(self.image_data) - 1

            img_emb = self.encode_images([img])[0]
            self.embeddings.append(img_emb)
            self.metadata.append(("image", img_idx))

            if caption:
                self.image_captions.append(caption)
                cap_emb = self.encode_texts([caption])[0]
                self.embeddings.append(cap_emb)
                self.metadata.append(("caption", img_idx))
            else:
                self.image_captions.append(None)

    def build_index(self):
        if self.embeddings:
            all_embs = np.vstack(self.embeddings).astype("float32")
            self.index = faiss.IndexFlatIP(all_embs.shape[1])
            self.index.add(all_embs)

    def encode_query(self, query_text=None, query_image=None):
        if query_text and query_image:
            text_emb = self.encode_texts([query_text])
            image_emb = self.encode_images([query_image])
            combined = text_emb + image_emb
            return combined / np.linalg.norm(combined, axis=1, keepdims=True)
        elif query_text:
            return self.encode_texts([query_text])
        elif query_image:
            return self.encode_images([query_image])
        else:
            raise ValueError("Provide text and/or image as query.")

    def retrieve(self, query_text=None, query_image=None, top_k=10):
        q_emb = self.encode_query(query_text, query_image)
        scores, indices = self.index.search(q_emb, top_k)
        ctx_texts, ctx_images = [], []
        for i in indices[0]:
            doc_type, doc_idx = self.metadata[i]
            if doc_type == "text":
                ctx_texts.append(self.text_data[doc_idx])
            elif doc_type == "caption":
                ctx_texts.append(self.image_captions[doc_idx])
                ctx_images.append(self.image_data[doc_idx])
            else:
                ctx_texts.append("[Image]")
                ctx_images.append(self.image_data[doc_idx])
        return ctx_texts, ctx_images

# ✅ Generator
class MultimodalGenerator:
    def __init__(self, model_name="llava-hf/llava-1.5-7b-hf"):
        self.processor = AutoProcessor.from_pretrained(model_name)
        self.model = LlavaForConditionalGeneration.from_pretrained(
            model_name, quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16), device_map="auto")

    def generate(self, query, query_image=None, context_texts=[], context_images=[]):
        img_list, question_string = [], query
        if query_image:
            question_string = f"<image>\n{query}"
            img_list.append(query_image)
        ctx = "\n".join(context_texts)
        if context_images:
            ctx = f"{'<image>'*len(context_images)}\n{ctx}"
            img_list.extend(context_images)
        prompt = f"USER: Context:{ctx}\n\nQuestion: {question_string}\n\nASSISTANT:"
        inputs = self.processor(img_list if img_list else prompt, prompt, padding=True, return_tensors="pt").to("cuda")
        output = self.model.generate(**inputs, max_new_tokens=200)
        return self.processor.batch_decode(output, skip_special_tokens=True)[0].split("ASSISTANT:")[-1]

# ✅ CLIP-based RAG Wrapper
class CLIPMultimodalRAG:
    def __init__(self):
        self.retriever = CLIPTextImageRetriever()
        self.generator = MultimodalGenerator()

    def add_texts(self, texts): self.retriever.add_texts(texts)
    def add_images(self, images, captions=None): self.retriever.add_images(images, captions)
    def build_index(self): self.retriever.build_index()

    def query(self, query_text=None, query_image=None, top_k=10):
        ctx_texts, ctx_images = self.retriever.retrieve(query_text, query_image, top_k)
        return self.generator.generate(query=query_text, query_image=query_image, context_texts=ctx_texts, context_images=ctx_images)


In [9]:


rag = CLIPMultimodalRAG()
rag.add_texts(rag_texts)
rag.add_images(rag_images)
rag.build_index()


# ✅ Evaluate without RAG
gen = MultimodalGenerator()
noctx_preds = [
    gen.generate(query=q, query_image=img).strip().lower()
    for q, img in zip(test_questions, test_images)
]


acc_simp_l_cap.append(compute_simple_accuracy(noctx_preds, test_answers)*100)
top_3_acc_l_cap.append(compute_topk_accuracy(noctx_preds, test_answers, k=3)*100)
# ✅ Report
print(f"✅ Non-RAG Accuracy: {compute_simple_accuracy(noctx_preds, test_answers)*100:.2f}%")
print(f"✅ Non-RAG Top-3 Accuracy: {compute_topk_accuracy(noctx_preds, test_answers, k=3)*100:.2f}%")


for k in k_ar[1:]:
  # ✅ Evaluate with RAG
  rag_preds = [
      rag.query(query_text=q, query_image=img, top_k=k).strip().lower()
      for q, img in zip(test_questions, test_images)]



  acc_simp_l_cap.append(compute_simple_accuracy(rag_preds, test_answers)*100)
  top_3_acc_l_cap.append(compute_topk_accuracy(rag_preds, test_answers, k=3)*100)
  # ✅ Report
  print(f"✅ RAG Accuracy: {compute_simple_accuracy(rag_preds, test_answers)*100:.2f}%")
  print(f"✅ RAG Top-3 Accuracy: {compute_topk_accuracy(rag_preds, test_answers, k=3)*100:.2f}%")


preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/990M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

✅ Non-RAG Accuracy: 0.00%
✅ Non-RAG Top-3 Accuracy: 31.00%
✅ RAG Accuracy: 21.00%
✅ RAG Top-3 Accuracy: 53.00%
✅ RAG Accuracy: 18.00%
✅ RAG Top-3 Accuracy: 52.00%
✅ RAG Accuracy: 24.00%
✅ RAG Top-3 Accuracy: 50.00%
✅ RAG Accuracy: 23.00%
✅ RAG Top-3 Accuracy: 43.00%
✅ RAG Accuracy: 15.00%
✅ RAG Top-3 Accuracy: 31.00%


In [10]:
acc_simp_l_cap

[0.0, 21.0, 18.0, 24.0, 23.0, 15.0]

In [11]:
top_3_acc_l_cap

[31.0, 53.0, 52.0, 50.0, 43.0, 31.0]